# Task 5 & 6: Feature Engineering Pipeline

**RecoMart E-commerce Recommendation System**

## Objective
Generate comprehensive feature tables from cleaned data for machine learning model training.

## Data Flow
```
INPUTS (workspace.recomart_silver):
├─ user_interaction_logs_cleaned
├─ purchase_history_cleaned
├─ product_catalog_cleaned
└─ external_signals_cleaned

OUTPUTS (workspace.recomart_silver):
├─ user_features              (User-level aggregated features)
├─ item_features              (Item-level aggregated features)
├─ user_item_features         (User-item interaction features + labels)
├─ co_purchase_pairs          (Frequently co-purchased item pairs)
└─ feature_metadata           (Feature catalog with descriptions)
```

## Feature Categories
1. **User Features**: Behavioral patterns, purchase history, preferences
2. **Item Features**: Popularity, quality, pricing, trends
3. **User-Item Features**: Interaction scores, purchase labels
4. **Co-Purchase Pairs**: Market basket analysis
5. **Feature Metadata**: Documentation for feature store

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

# Configuration
CATALOG = "workspace"
SCHEMA = "recomart_silver"
REFERENCE_DATE = datetime(2026, 4, 26)  # Today's date for recency calculations

print("✅ Imports loaded")
print(f"   Catalog: {CATALOG}")
print(f"   Schema: {SCHEMA}")
print(f"   Reference date: {REFERENCE_DATE.strftime('%Y-%m-%d')}")

In [0]:
print("Loading cleaned data from Unity Catalog...\n")

# Load cleaned tables
df_clicks = spark.table(f"{CATALOG}.{SCHEMA}.user_interaction_logs_cleaned")
df_txn = spark.table(f"{CATALOG}.{SCHEMA}.purchase_history_cleaned")
df_catalog = spark.table(f"{CATALOG}.{SCHEMA}.product_catalog_cleaned")
df_external = spark.table(f"{CATALOG}.{SCHEMA}.external_signals_cleaned")

print(f"✅ user_interaction_logs_cleaned: {df_clicks.count():,} rows")
print(f"✅ purchase_history_cleaned:      {df_txn.count():,} rows")
print(f"✅ product_catalog_cleaned:       {df_catalog.count():,} rows")
print(f"✅ external_signals_cleaned:      {df_external.count():,} rows")

In [0]:
def min_max_normalize(df, col, new_col=None):
    """Min-Max normalization: scales values to [0, 1]"""
    if new_col is None:
        new_col = f"{col}_norm"
    
    min_val = df.agg(F.min(col)).collect()[0][0]
    max_val = df.agg(F.max(col)).collect()[0][0]
    
    if max_val == min_val:
        return df.withColumn(new_col, F.lit(0.0))
    
    return df.withColumn(new_col, 
        (F.col(col) - F.lit(min_val)) / (F.lit(max_val) - F.lit(min_val)))

def log_transform(df, col, new_col=None):
    """Log transformation: log(x + 1) to handle zeros"""
    if new_col is None:
        new_col = f"{col}_log"
    return df.withColumn(new_col, F.log1p(F.col(col)))

def label_encode(df, col, new_col=None):
    """Simple label encoding using dense_rank"""
    if new_col is None:
        new_col = f"{col}_enc"
    
    window = Window.orderBy(col)
    return df.withColumn(new_col, F.dense_rank().over(window) - 1)

print("✅ Helper functions defined")

In [0]:
print("="*60)
print("GENERATING USER FEATURES")
print("="*60)

# Get unique users from clickstream
users = df_clicks.select("user_id").distinct()

# 1. Clickstream aggregations
user_click_agg = df_clicks.groupBy("user_id").agg(
    F.count("*").alias("user_total_events"),
    F.countDistinct("item_id").alias("user_distinct_items"),
    F.max(F.to_timestamp("timestamp")).alias("user_last_event"),
    F.min(F.to_timestamp("timestamp")).alias("user_first_event"),
    F.mode("device").alias("user_preferred_device"),
    F.mode("event_type").alias("user_preferred_event_type"),
    F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("user_view_count"),
    F.sum(F.when(F.col("event_type") == "click", 1).otherwise(0)).alias("user_click_count"),
    F.sum(F.when(F.col("event_type") == "add_to_cart", 1).otherwise(0)).alias("user_cart_count"),
    F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("user_purchase_count_events")
)

# 2. Transaction aggregations - join with catalog to get category
df_txn_with_cat = df_txn.join(df_catalog.select("item_id", "category"), "item_id", "left")

user_txn_agg = df_txn_with_cat.groupBy("user_id").agg(
    F.count("*").alias("user_purchase_count"),
    F.sum("total_price").alias("user_total_spend"),
    F.avg("total_price").alias("user_avg_spend"),
    F.max("total_price").alias("user_max_spend"),
    F.sum("quantity").alias("user_total_quantity"),
    F.countDistinct("item_id").alias("user_distinct_purchased_items"),
    F.mode("payment_method").alias("user_preferred_payment"),
    F.max(F.to_timestamp("purchase_date")).alias("user_last_purchase"),
    F.mode("category").alias("user_preferred_category")
)

# 3. Join clickstream and transaction features
user_features = users.join(user_click_agg, "user_id", "left") \
                     .join(user_txn_agg, "user_id", "left")

# Fill nulls for users with no purchases
user_features = user_features.fillna({
    "user_purchase_count": 0,
    "user_total_spend": 0.0,
    "user_avg_spend": 0.0,
    "user_max_spend": 0.0,
    "user_total_quantity": 0,
    "user_distinct_purchased_items": 0
})

# 4. Derived features
user_features = user_features.withColumn(
    "user_purchase_rate",
    F.when(F.col("user_total_events") > 0, 
           F.col("user_purchase_count") / F.col("user_total_events")).otherwise(0.0)
)

user_features = user_features.withColumn(
    "user_recency_days",
    F.datediff(F.lit(REFERENCE_DATE), F.coalesce(F.col("user_last_purchase"), F.col("user_last_event")))
)

user_features = user_features.withColumn(
    "user_tenure_days",
    F.datediff(F.col("user_last_event"), F.col("user_first_event")) + 1
)

user_features = user_features.withColumn(
    "user_activity_score",
    F.col("user_total_events") + (F.col("user_purchase_count") * 5)
)

# 5. Label encoding for categorical features
user_features = label_encode(user_features, "user_preferred_device", "user_preferred_device_enc")
user_features = label_encode(user_features, "user_preferred_category", "user_preferred_category_enc")

# 6. Normalize continuous features
user_features = min_max_normalize(user_features, "user_activity_score", "user_activity_score_norm")
user_features = log_transform(user_features, "user_total_spend", "user_total_spend_log")

# Select final columns
user_features = user_features.select(
    "user_id",
    "user_total_events",
    "user_distinct_items",
    "user_purchase_count",
    "user_purchase_rate",
    "user_total_spend",
    "user_total_spend_log",
    "user_avg_spend",
    "user_max_spend",
    "user_total_quantity",
    "user_distinct_purchased_items",
    "user_recency_days",
    "user_tenure_days",
    "user_activity_score",
    "user_activity_score_norm",
    "user_preferred_device",
    "user_preferred_device_enc",
    "user_preferred_category",
    "user_preferred_category_enc",
    "user_view_count",
    "user_click_count",
    "user_cart_count"
)

print(f"\n✅ User features generated: {user_features.count():,} users")
print(f"   Columns: {len(user_features.columns)}")
user_features.show(3, truncate=False)

In [0]:
print("\n" + "="*60)
print("GENERATING ITEM FEATURES")
print("="*60)

# Get all items from catalog
items = df_catalog.select("item_id").distinct()

# 1. Clickstream aggregations (item popularity)
item_click_agg = df_clicks.groupBy("item_id").agg(
    F.count("*").alias("item_total_interactions"),
    F.countDistinct("user_id").alias("item_unique_users"),
    F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("item_view_count"),
    F.sum(F.when(F.col("event_type") == "click", 1).otherwise(0)).alias("item_click_count"),
    F.sum(F.when(F.col("event_type") == "add_to_cart", 1).otherwise(0)).alias("item_cart_count"),
    F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("item_purchase_count_events")
)

# 2. Transaction aggregations
item_txn_agg = df_txn.groupBy("item_id").agg(
    F.count("*").alias("item_purchase_count"),
    F.sum("quantity").alias("item_total_quantity_sold"),
    F.sum("total_price").alias("item_total_revenue"),
    F.avg("total_price").alias("item_avg_order_value"),
    F.countDistinct("user_id").alias("item_unique_buyers")
)

# 3. Catalog features
item_catalog = df_catalog.select(
    "item_id",
    "category",
    "brand",
    "price",
    "stock",
    "rating",
    "review_count",
    "is_active"
)

# 4. External signals
item_external = df_external.select(
    "item_id",
    "popularity_score",
    "avg_sentiment",
    "trend_score",
    "return_rate"
)

# 5. Join all item features
item_features = items.join(item_click_agg, "item_id", "left") \
                     .join(item_txn_agg, "item_id", "left") \
                     .join(item_catalog, "item_id", "left") \
                     .join(item_external, "item_id", "left")

# Fill nulls for items with no interactions
item_features = item_features.fillna({
    "item_total_interactions": 0,
    "item_unique_users": 0,
    "item_view_count": 0,
    "item_click_count": 0,
    "item_cart_count": 0,
    "item_purchase_count_events": 0,
    "item_purchase_count": 0,
    "item_total_quantity_sold": 0,
    "item_total_revenue": 0.0,
    "item_avg_order_value": 0.0,
    "item_unique_buyers": 0
})

# 6. Derived features
item_features = item_features.withColumn(
    "item_conversion_rate",
    F.when(F.col("item_total_interactions") > 0,
           F.col("item_purchase_count") / F.col("item_total_interactions")).otherwise(0.0)
)

item_features = item_features.withColumn(
    "item_avg_rating",
    F.coalesce(F.col("rating"), F.lit(0.0))
)

item_features = item_features.withColumn(
    "item_avg_sentiment",
    F.coalesce(F.col("avg_sentiment"), F.lit(0.0))
)

item_features = item_features.withColumn(
    "item_return_rate",
    F.coalesce(F.col("return_rate"), F.lit(0.0))
)

# 7. Label encoding for categorical features
item_features = label_encode(item_features, "category", "item_category_enc")
item_features = label_encode(item_features, "brand", "item_brand_enc")

# 8. Normalize and transform continuous features
item_features = min_max_normalize(item_features, "popularity_score", "item_popularity_score_norm")
item_features = min_max_normalize(item_features, "trend_score", "item_trend_score_norm")
item_features = min_max_normalize(item_features, "stock", "item_stock_norm")
item_features = log_transform(item_features, "price", "item_price_log")
item_features = log_transform(item_features, "review_count", "item_review_count_log")

# Select final columns
item_features = item_features.select(
    "item_id",
    "item_total_interactions",
    "item_unique_users",
    "item_purchase_count",
    "item_conversion_rate",
    "item_total_quantity_sold",
    "item_total_revenue",
    "item_avg_order_value",
    "item_unique_buyers",
    "item_avg_rating",
    "item_view_count",
    "item_click_count",
    "item_cart_count",
    "category",
    "item_category_enc",
    "brand",
    "item_brand_enc",
    "price",
    "item_price_log",
    "stock",
    "item_stock_norm",
    "review_count",
    "item_review_count_log",
    "popularity_score",
    "item_popularity_score_norm",
    "item_avg_sentiment",
    "trend_score",
    "item_trend_score_norm",
    "item_return_rate",
    "is_active"
)

print(f"\n✅ Item features generated: {item_features.count():,} items")
print(f"   Columns: {len(item_features.columns)}")
item_features.show(3, truncate=False)

In [0]:
print("\n" + "="*60)
print("GENERATING USER-ITEM FEATURES")
print("="*60)

# 1. Create user-item pairs from clickstream
user_item_clicks = df_clicks.groupBy("user_id", "item_id").agg(
    F.count("*").alias("ui_total_interactions"),
    F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("ui_views"),
    F.sum(F.when(F.col("event_type") == "click", 1).otherwise(0)).alias("ui_clicks"),
    F.sum(F.when(F.col("event_type") == "add_to_cart", 1).otherwise(0)).alias("ui_carts"),
    F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("ui_purchases_events")
)

# 2. Check if user actually purchased the item (from transactions)
user_item_purchases = df_txn.select("user_id", "item_id") \
    .withColumn("ui_has_purchased", F.lit(1)) \
    .distinct()

# 3. Join and create label
user_item_features = user_item_clicks.join(
    user_item_purchases, 
    ["user_id", "item_id"], 
    "left"
).fillna({"ui_has_purchased": 0})

# 4. Calculate interaction score (weighted)
user_item_features = user_item_features.withColumn(
    "ui_interaction_score",
    (F.col("ui_views") * 1) + 
    (F.col("ui_clicks") * 2) + 
    (F.col("ui_carts") * 3) + 
    (F.col("ui_purchases_events") * 5)
)

# 5. Normalize interaction score
user_item_features = min_max_normalize(
    user_item_features, 
    "ui_interaction_score", 
    "ui_interaction_score_norm"
)

# Select final columns
user_item_features = user_item_features.select(
    "user_id",
    "item_id",
    "ui_total_interactions",
    "ui_views",
    "ui_clicks",
    "ui_carts",
    "ui_purchases_events",
    "ui_interaction_score",
    "ui_interaction_score_norm",
    "ui_has_purchased"
)

print(f"\n✅ User-Item features generated: {user_item_features.count():,} pairs")
print(f"   Columns: {len(user_item_features.columns)}")
print(f"\n   Label distribution:")
user_item_features.groupBy("ui_has_purchased").count().show()
user_item_features.show(3, truncate=False)

In [0]:
print("\n" + "="*60)
print("GENERATING CO-PURCHASE PAIRS")
print("="*60)

# 1. Get items purchased in same order
orders_with_items = df_txn.select("order_id", "item_id")

# 2. Self-join to find item pairs in same order
item_pairs = orders_with_items.alias("a").join(
    orders_with_items.alias("b"),
    (F.col("a.order_id") == F.col("b.order_id")) & 
    (F.col("a.item_id") < F.col("b.item_id"))  # Avoid duplicates and self-pairs
).select(
    F.col("a.item_id").alias("item_a"),
    F.col("b.item_id").alias("item_b"),
    F.col("a.order_id")
)

# 3. Count co-occurrences
co_purchase_counts = item_pairs.groupBy("item_a", "item_b").agg(
    F.count("*").alias("co_purchase_count")
)

# 4. Calculate support (how often the pair appears)
total_orders = df_txn.select("order_id").distinct().count()

co_purchase_pairs = co_purchase_counts.withColumn(
    "support",
    F.col("co_purchase_count") / F.lit(total_orders)
)

# 5. Calculate confidence and lift
# Get individual item purchase counts
item_purchase_counts = df_txn.groupBy("item_id").agg(
    F.countDistinct("order_id").alias("item_order_count")
)

co_purchase_pairs = co_purchase_pairs.join(
    item_purchase_counts.select(
        F.col("item_id").alias("item_a"),
        F.col("item_order_count").alias("item_a_count")
    ),
    "item_a"
).join(
    item_purchase_counts.select(
        F.col("item_id").alias("item_b"),
        F.col("item_order_count").alias("item_b_count")
    ),
    "item_b"
)

co_purchase_pairs = co_purchase_pairs.withColumn(
    "confidence_a_to_b",
    F.col("co_purchase_count") / F.col("item_a_count")
).withColumn(
    "confidence_b_to_a",
    F.col("co_purchase_count") / F.col("item_b_count")
).withColumn(
    "lift",
    (F.col("co_purchase_count") * F.lit(total_orders)) / 
    (F.col("item_a_count") * F.col("item_b_count"))
)

# 6. Filter for meaningful pairs (minimum support and lift)
co_purchase_pairs = co_purchase_pairs.filter(
    (F.col("co_purchase_count") >= 2) &  # At least 2 co-purchases
    (F.col("lift") > 1.0)  # Lift > 1 means positive association
)

# Select final columns
co_purchase_pairs = co_purchase_pairs.select(
    "item_a",
    "item_b",
    "co_purchase_count",
    "support",
    "confidence_a_to_b",
    "confidence_b_to_a",
    "lift"
).orderBy(F.desc("lift"))

print(f"\n✅ Co-purchase pairs generated: {co_purchase_pairs.count():,} pairs")
print(f"   Columns: {len(co_purchase_pairs.columns)}")
print(f"\n   Top 5 pairs by lift:")
co_purchase_pairs.show(5, truncate=False)

In [0]:
print("\n" + "="*60)
print("GENERATING FEATURE METADATA")
print("="*60)

# Define feature metadata
metadata_rows = [
    # User features
    ("user_total_events", "user", "user_features", "user_id", "long", "continuous", "Total number of events (views, clicks, carts, purchases)", "clickstream", "v1.0"),
    ("user_distinct_items", "user", "user_features", "user_id", "long", "continuous", "Number of distinct items interacted with", "clickstream", "v1.0"),
    ("user_purchase_count", "user", "user_features", "user_id", "long", "continuous", "Total number of purchases made", "transactions", "v1.0"),
    ("user_purchase_rate", "user", "user_features", "user_id", "double", "continuous", "Purchase conversion rate (purchases / total events)", "derived", "v1.0"),
    ("user_total_spend", "user", "user_features", "user_id", "double", "continuous", "Total amount spent by user", "transactions", "v1.0"),
    ("user_total_spend_log", "user", "user_features", "user_id", "double", "continuous", "Log-transformed total spend", "derived", "v1.0"),
    ("user_avg_spend", "user", "user_features", "user_id", "double", "continuous", "Average spend per purchase", "transactions", "v1.0"),
    ("user_max_spend", "user", "user_features", "user_id", "double", "continuous", "Maximum single purchase amount", "transactions", "v1.0"),
    ("user_total_quantity", "user", "user_features", "user_id", "long", "continuous", "Total quantity of items purchased", "transactions", "v1.0"),
    ("user_distinct_purchased_items", "user", "user_features", "user_id", "long", "continuous", "Number of distinct items purchased", "transactions", "v1.0"),
    ("user_recency_days", "user", "user_features", "user_id", "long", "continuous", "Days since last activity", "derived", "v1.0"),
    ("user_tenure_days", "user", "user_features", "user_id", "long", "continuous", "Days between first and last event", "derived", "v1.0"),
    ("user_activity_score", "user", "user_features", "user_id", "long", "continuous", "Weighted activity score (events + 5*purchases)", "derived", "v1.0"),
    ("user_activity_score_norm", "user", "user_features", "user_id", "double", "continuous", "Normalized activity score [0,1]", "derived", "v1.0"),
    ("user_preferred_device", "user", "user_features", "user_id", "string", "categorical", "Most frequently used device", "clickstream", "v1.0"),
    ("user_preferred_device_enc", "user", "user_features", "user_id", "long", "categorical", "Label-encoded preferred device", "derived", "v1.0"),
    ("user_preferred_category", "user", "user_features", "user_id", "string", "categorical", "Most frequently purchased category", "transactions", "v1.0"),
    ("user_preferred_category_enc", "user", "user_features", "user_id", "long", "categorical", "Label-encoded preferred category", "derived", "v1.0"),
    ("user_view_count", "user", "user_features", "user_id", "long", "continuous", "Total view events", "clickstream", "v1.0"),
    ("user_click_count", "user", "user_features", "user_id", "long", "continuous", "Total click events", "clickstream", "v1.0"),
    ("user_cart_count", "user", "user_features", "user_id", "long", "continuous", "Total add-to-cart events", "clickstream", "v1.0"),
    
    # Item features
    ("item_total_interactions", "item", "item_features", "item_id", "long", "continuous", "Total number of interactions with item", "clickstream", "v1.0"),
    ("item_unique_users", "item", "item_features", "item_id", "long", "continuous", "Number of unique users who interacted", "clickstream", "v1.0"),
    ("item_purchase_count", "item", "item_features", "item_id", "long", "continuous", "Total number of purchases", "transactions", "v1.0"),
    ("item_conversion_rate", "item", "item_features", "item_id", "double", "continuous", "Purchase conversion rate (purchases / interactions)", "derived", "v1.0"),
    ("item_total_quantity_sold", "item", "item_features", "item_id", "long", "continuous", "Total quantity sold", "transactions", "v1.0"),
    ("item_total_revenue", "item", "item_features", "item_id", "double", "continuous", "Total revenue generated", "transactions", "v1.0"),
    ("item_avg_order_value", "item", "item_features", "item_id", "double", "continuous", "Average order value", "transactions", "v1.0"),
    ("item_unique_buyers", "item", "item_features", "item_id", "long", "continuous", "Number of unique buyers", "transactions", "v1.0"),
    ("item_avg_rating", "item", "item_features", "item_id", "double", "continuous", "Average customer rating", "catalog", "v1.0"),
    ("item_view_count", "item", "item_features", "item_id", "long", "continuous", "Total view events", "clickstream", "v1.0"),
    ("item_click_count", "item", "item_features", "item_id", "long", "continuous", "Total click events", "clickstream", "v1.0"),
    ("item_cart_count", "item", "item_features", "item_id", "long", "continuous", "Total add-to-cart events", "clickstream", "v1.0"),
    ("category", "item", "item_features", "item_id", "string", "categorical", "Product category", "catalog", "v1.0"),
    ("item_category_enc", "item", "item_features", "item_id", "long", "categorical", "Label-encoded category", "derived", "v1.0"),
    ("brand", "item", "item_features", "item_id", "string", "categorical", "Product brand", "catalog", "v1.0"),
    ("item_brand_enc", "item", "item_features", "item_id", "long", "categorical", "Label-encoded brand", "derived", "v1.0"),
    ("price", "item", "item_features", "item_id", "double", "continuous", "Item price", "catalog", "v1.0"),
    ("item_price_log", "item", "item_features", "item_id", "double", "continuous", "Log-transformed price", "derived", "v1.0"),
    ("stock", "item", "item_features", "item_id", "long", "continuous", "Current stock level", "catalog", "v1.0"),
    ("item_stock_norm", "item", "item_features", "item_id", "double", "continuous", "Normalized stock level [0,1]", "derived", "v1.0"),
    ("review_count", "item", "item_features", "item_id", "long", "continuous", "Number of reviews", "catalog", "v1.0"),
    ("item_review_count_log", "item", "item_features", "item_id", "double", "continuous", "Log-transformed review count", "derived", "v1.0"),
    ("popularity_score", "item", "item_features", "item_id", "double", "continuous", "External popularity score", "external", "v1.0"),
    ("item_popularity_score_norm", "item", "item_features", "item_id", "double", "continuous", "Normalized popularity score [0,1]", "derived", "v1.0"),
    ("item_avg_sentiment", "item", "item_features", "item_id", "double", "continuous", "Average sentiment score", "external", "v1.0"),
    ("trend_score", "item", "item_features", "item_id", "double", "continuous", "Trending score", "external", "v1.0"),
    ("item_trend_score_norm", "item", "item_features", "item_id", "double", "continuous", "Normalized trend score [0,1]", "derived", "v1.0"),
    ("item_return_rate", "item", "item_features", "item_id", "double", "continuous", "Product return rate", "external", "v1.0"),
    
    # User-Item features
    ("ui_total_interactions", "user_item", "user_item_features", "user_id", "long", "continuous", "Total interactions between user and item", "clickstream", "v1.0"),
    ("ui_views", "user_item", "user_item_features", "user_id", "long", "continuous", "Number of views", "clickstream", "v1.0"),
    ("ui_clicks", "user_item", "user_item_features", "user_id", "long", "continuous", "Number of clicks", "clickstream", "v1.0"),
    ("ui_carts", "user_item", "user_item_features", "user_id", "long", "continuous", "Number of add-to-cart events", "clickstream", "v1.0"),
    ("ui_purchases_events", "user_item", "user_item_features", "user_id", "long", "continuous", "Number of purchase events", "clickstream", "v1.0"),
    ("ui_interaction_score", "user_item", "user_item_features", "user_id", "long", "continuous", "Weighted interaction score", "derived", "v1.0"),
    ("ui_interaction_score_norm", "user_item", "user_item_features", "user_id", "double", "continuous", "Normalized interaction score [0,1]", "derived", "v1.0"),
    ("ui_has_purchased", "user_item", "user_item_features", "user_id", "long", "binary", "Purchase label (0=no, 1=yes)", "transactions", "v1.0"),
]

metadata_schema = StructType([
    StructField("feature_name", StringType(), False),
    StructField("group", StringType(), False),
    StructField("table", StringType(), False),
    StructField("primary_key", StringType(), False),
    StructField("dtype", StringType(), False),
    StructField("feature_type", StringType(), False),
    StructField("description", StringType(), False),
    StructField("source", StringType(), False),
    StructField("version", StringType(), False)
])

feature_metadata = spark.createDataFrame(metadata_rows, metadata_schema)

print(f"\n✅ Feature metadata generated: {feature_metadata.count()} features")
print(f"\n   Feature breakdown by group:")
feature_metadata.groupBy("group").count().orderBy("group").show()
print(f"\n   Sample metadata:")
feature_metadata.show(5, truncate=False)

In [0]:
print("\n" + "="*60)
print("SAVING FEATURE TABLES TO UNITY CATALOG")
print("="*60)

# Save user features
print(f"\n[1/5] Saving user_features...")
user_features.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.user_features")
print(f"✅ Saved: {CATALOG}.{SCHEMA}.user_features ({user_features.count():,} rows)")

# Save item features
print(f"\n[2/5] Saving item_features...")
item_features.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.item_features")
print(f"✅ Saved: {CATALOG}.{SCHEMA}.item_features ({item_features.count():,} rows)")

# Save user-item features
print(f"\n[3/5] Saving user_item_features...")
user_item_features.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.user_item_features")
print(f"✅ Saved: {CATALOG}.{SCHEMA}.user_item_features ({user_item_features.count():,} rows)")

# Save co-purchase pairs
print(f"\n[4/5] Saving co_purchase_pairs...")
co_purchase_pairs.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.co_purchase_pairs")
print(f"✅ Saved: {CATALOG}.{SCHEMA}.co_purchase_pairs ({co_purchase_pairs.count():,} rows)")

# Save feature metadata
print(f"\n[5/5] Saving feature_metadata...")
feature_metadata.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.feature_metadata")
print(f"✅ Saved: {CATALOG}.{SCHEMA}.feature_metadata ({feature_metadata.count()} rows)")

print("\n" + "="*60)
print("ALL FEATURE TABLES SAVED SUCCESSFULLY")
print("="*60)

In [0]:
print("\n" + "="*60)
print("FEATURE ENGINEERING PIPELINE SUMMARY")
print("="*60)

print("\n📊 FEATURE TABLE SUMMARY:")
print(f"   ├─ user_features:        {user_features.count():>6,} rows × {len(user_features.columns):>2} columns")
print(f"   ├─ item_features:        {item_features.count():>6,} rows × {len(item_features.columns):>2} columns")
print(f"   ├─ user_item_features:   {user_item_features.count():>6,} rows × {len(user_item_features.columns):>2} columns")
print(f"   ├─ co_purchase_pairs:    {co_purchase_pairs.count():>6,} rows × {len(co_purchase_pairs.columns):>2} columns")
print(f"   └─ feature_metadata:     {feature_metadata.count():>6} rows × {len(feature_metadata.columns):>2} columns")

print("\n📈 SAMPLE PREVIEWS:")

print("\n[User Features Sample]")
user_features.select(
    "user_id", 
    "user_total_events", 
    "user_purchase_count", 
    "user_purchase_rate",
    "user_activity_score_norm"
).show(3, truncate=False)

print("\n[Item Features Sample]")
item_features.select(
    "item_id", 
    "item_total_interactions", 
    "item_purchase_count",
    "item_conversion_rate",
    "item_avg_rating"
).show(3, truncate=False)

print("\n[User-Item Features Sample]")
user_item_features.select(
    "user_id",
    "item_id",
    "ui_interaction_score",
    "ui_interaction_score_norm",
    "ui_has_purchased"
).show(3, truncate=False)

print("\n[Co-Purchase Pairs Sample]")
co_purchase_pairs.select(
    "item_a",
    "item_b",
    "co_purchase_count",
    "lift"
).show(3, truncate=False)

print("\n✅ Feature Engineering Pipeline Complete!")
print("   All tables saved to workspace.recomart_silver")
print("   Ready for Task 7 (Feature Store) and Task 8 (Model Training)")

In [0]:
print("\n" + "="*60)
print("EXPORTING FEATURES TO CSV FOR TASK 7")
print("="*60)

# Define output directory
output_dir = "/Workspace/Users/2025ae05764@wilp.bits-pilani.ac.in/RecomartEcommerce/2_medallion_processing/features"

# Convert to Pandas and export
print(f"\n[1/5] Exporting user_features.csv...")
user_features.toPandas().to_csv(f"{output_dir}/user_features.csv", index=False)
print(f"✅ Exported: {output_dir}/user_features.csv")

print(f"\n[2/5] Exporting item_features.csv...")
item_features.toPandas().to_csv(f"{output_dir}/item_features.csv", index=False)
print(f"✅ Exported: {output_dir}/item_features.csv")

print(f"\n[3/5] Exporting user_item_features.csv...")
user_item_features.toPandas().to_csv(f"{output_dir}/user_item_features.csv", index=False)
print(f"✅ Exported: {output_dir}/user_item_features.csv")

print(f"\n[4/5] Exporting co_purchase_pairs.csv...")
co_purchase_pairs.toPandas().to_csv(f"{output_dir}/co_purchase_pairs.csv", index=False)
print(f"✅ Exported: {output_dir}/co_purchase_pairs.csv")

print(f"\n[5/5] Exporting feature_metadata.csv...")
feature_metadata.toPandas().to_csv(f"{output_dir}/feature_metadata.csv", index=False)
print(f"✅ Exported: {output_dir}/feature_metadata.csv")

print("\n" + "="*60)
print("ALL FEATURES EXPORTED TO CSV")
print("="*60)
print(f"\nLocation: {output_dir}/")
print("Ready for Task 7 Feature Store notebook!")